# Park-Vision — Fine-tune on the Roadside Parking dataset (300 images)

This notebook:
1. Clones the [Park-Vision](https://github.com/jawadstalker/Park-Vision) repo
2. Downloads the [Roadside Parking dataset](https://universe.roboflow.com/3883zn-gmail-com/roadside-parking) from Roboflow (street-level images, closest public match to this project's scenario)
3. Samples 300 images, keeping only the `car` class
4. Splits them into train/val/test
5. Fine-tunes YOLOv8 and evaluates it against the project's acceptance criteria

You need a free [Roboflow](https://roboflow.com) account and API key (Settings → API Keys) to run the download cell — even public Universe datasets require one.


## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/jawadstalker/Park-Vision.git
%cd Park-Vision
!pip install -q -r requirements.txt
!pip install -q roboflow pyyaml


## 2. Download the dataset from Roboflow

Paste your API key when prompted (it is not stored anywhere, only used for this session).
If the download fails with a "version not found" error, open the dataset page, click
**Export**, and copy the exact `workspace/project/version` shown there — Roboflow
occasionally publishes a newer version number than the one used below.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("3883zn-gmail-com").project("roadside-parking")
dataset = project.version(1).download("yolov8", location="/content/roadside_parking_raw")


## 3. Convert to this project's format and sample 300 images

Drops the `motorcycle` class, remaps `car` to class id 0, and spreads the 300
sampled images across 20 pseudo-street tags so the train/val/test split isn't
lumped into a single bucket (see `dataset_tools/README.md` for why).


In [ ]:
!python dataset_tools/prepare_external_dataset.py \
    /content/roadside_parking_raw /content/raw_frames /content/raw_labels \
    --source-name roadsideparking --lighting day --keep-class car \
    --chunks 20 --limit 300


## 4. Check the collected distribution, then split train/val/test

In [ ]:
!python dataset_tools/dataset_stats.py /content/raw_frames


In [ ]:
!python dataset_tools/organize_dataset.py \
    /content/raw_frames /content/raw_labels /content/dataset \
    --test-ratio 0.15 --val-ratio 0.15


## 5. Fine-tune YOLOv8

Uses the free Colab GPU if you enabled one (Runtime → Change runtime type → GPU).
50 epochs is enough for a quick pipeline check on 300 images; raise it for a
more serious run once you're validating with real Mashhad footage.


In [ ]:
import torch
device = "0" if torch.cuda.is_available() else "cpu"
print("Training on device:", device)

!python dataset_tools/train.py /content/dataset/data.yaml \
    --model yolov8n.pt --epochs 50 --device {device} \
    --project /content/runs --name street_parking


## 6. Evaluate against the project's acceptance criteria

Checks Precision >= 85%, Recall >= 80%, mAP@0.5 >= 80%, and per-frame
inference time < 3s (the CPU/no-GPU target from the feasibility document —
timing will look better here on Colab's GPU, so re-check on your own CPU
hardware before trusting the timing numbers for the final report).


In [ ]:
!python dataset_tools/evaluate.py \
    /content/runs/street_parking/weights/best.pt \
    /content/dataset/data.yaml --device {device}


## 7. Try it on a single test image

Runs the fine-tuned weights through the same `VehicleDetector` wrapper the
FastAPI service uses, so the result matches what `/detect` would return.


In [ ]:
import glob
import cv2
from app.vehicle_detector import VehicleDetector

test_images = glob.glob("/content/dataset/images/test/*.jpg")
print(f"{len(test_images)} test images available")

detector = VehicleDetector(model_path="/content/runs/street_parking/weights/best.pt", device=device)
sample = cv2.imread(test_images[0])
detections = detector.detect(sample)
print(f"Detected {len(detections)} vehicles in {test_images[0]}")
detections


## Next steps

- This dataset is supplementary training data, not a substitute for the
  project's required field test (>=500 images from 3 separate Mashhad streets).
- Once you have real street footage, run `extract_frames.py` on it and merge
  it in with `organize_dataset.py` alongside this data, then re-run steps 5-6.
- Download `/content/runs/street_parking/weights/best.pt` and drop it into
  `app/vehicle_detector.py`'s `model_path` to use it in the live API.
